In [1]:
from langgraph.graph import StateGraph, START, END
from typing import Annotated, TypedDict
import operator
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
# Build the graph
class State(TypedDict):
    data: Annotated[str, operator.add]   

def node_a(state: State) -> State:
    print("Node A executed")
    return {"data": "A "}

def node_b(state: State) -> State:
    print("Node B executed")
    return {"data": "B "}

def node_c(state: State) -> State:
    print("Node C executed")
    return {"data": "C "}

def node_d(state: State) -> State:
    print("Node D executed")
    return {"data": "D "}

def node_e(state: State) -> State:
    print("Node E executed")
    return {"data": "E "}

graph_builder = StateGraph(State)

graph_builder.add_node("A", node_a)
graph_builder.add_node("B", node_b)
graph_builder.add_node("C", node_c)
graph_builder.add_node("D", node_d)
graph_builder.add_node("E", node_e)

graph_builder.add_edge(START, "A")
graph_builder.add_edge("A", "B")
graph_builder.add_edge("B", "C")
graph_builder.add_edge("B", "D")
graph_builder.add_edge("C", "E")
graph_builder.add_edge("D", "E")
graph_builder.add_edge("E", END)

In [3]:
# Create an InMemorySaver checkpointer
memory = InMemorySaver()

# Compile the graph with the checkpointer enabled
graph = graph_builder.compile(checkpointer=memory)

In [4]:
# # Define a runtime configuration with a thread_id
config = {"configurable": {"thread_id": "demo-thread-1"}}

# Invoke the graph with config
final_state = graph.invoke({"data": ""}, config=config)
print("Final state:", final_state)

Node A executed
Node B executed
Node C executed
Node D executed
Node E executed
Final state: {'data': 'A B C D E '}


In [5]:
# Retrieve the latest checkpoint (state snapshot) for this thread
graph.get_state(config)

StateSnapshot(values={'data': 'A B C D E '}, next=(), config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1796d5-7d85-66fb-8004-fe2e825d436d'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-07-06T19:03:15.738776+00:00', parent_config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1796d5-7d7e-656e-8003-a6e02867e24d'}}, tasks=(), interrupts=())

In [6]:
# Access only the state values from the latest checkpoint
graph.get_state(config).values

{'data': 'A B C D E '}

In [7]:
# Retrieve the full checkpoint history for this thread
list(graph.get_state_history(config))

[StateSnapshot(values={'data': 'A B C D E '}, next=(), config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1796d5-7d85-66fb-8004-fe2e825d436d'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-07-06T19:03:15.738776+00:00', parent_config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1796d5-7d7e-656e-8003-a6e02867e24d'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'data': 'A B C D '}, next=('E',), config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1796d5-7d7e-656e-8003-a6e02867e24d'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-07-06T19:03:15.735823+00:00', parent_config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1796d5-7d62-62f2-8002-fce8ef298f3a'}}, tasks=(PregelTask(id='86e14266-acb7-25b1-62e6-da5c85062beb', name='E', path=('__pregel_pull', 'E'), erro

In [8]:
# Invoke the graph again using the SAME thread_id, workflow resumes from the last checkpoint
config = {"configurable": {"thread_id": "demo-thread-1"}}
final_state = graph.invoke({"data": "New "}, config=config)
print("Final state:", final_state)

Node A executed
Node B executed
Node C executed
Node D executed
Node E executed
Final state: {'data': 'A B C D E New A B C D E '}


In [9]:
# View the updated checkpoint history, this now includes checkpoints from both executions
list(graph.get_state_history(config))

[StateSnapshot(values={'data': 'A B C D E New A B C D E '}, next=(), config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1796de-4a21-6191-800a-a1f3efb8610b'}}, metadata={'source': 'loop', 'step': 10, 'parents': {}}, created_at='2026-07-06T19:07:11.941819+00:00', parent_config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1796de-4a1a-6375-8009-bd308a2022cd'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'data': 'A B C D E New A B C D '}, next=('E',), config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1796de-4a1a-6375-8009-bd308a2022cd'}}, metadata={'source': 'loop', 'step': 9, 'parents': {}}, created_at='2026-07-06T19:07:11.938974+00:00', parent_config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1796de-4a12-65f1-8008-ce0b4a259a23'}}, tasks=(PregelTask(id='4a18c211-b55c-916c-8c0f-3f4967c0b21a', name='E', path

In [10]:
# Invoke the graph with a DIFFERENT thread_id, This creates a completely new, independent execution
final_state = graph.invoke({"data": "222 "}, config={"configurable": {"thread_id": "demo-thread-2"}})
print("Final state:", final_state)

Node A executed
Node B executed
Node C executed
Node D executed
Node E executed
Final state: {'data': '222 A B C D E '}


In [11]:
# Retrieve checkpoint history for demo-thread-2, this is separate from demo-thread-1
list(graph.get_state_history(config={"configurable": {"thread_id": "demo-thread-2"}}))

[StateSnapshot(values={'data': '222 A B C D E '}, next=(), config={'configurable': {'thread_id': 'demo-thread-2', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb542-62c4-6ace-8004-4019d1df96ec'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-01-06T23:05:09.870254+00:00', parent_config={'configurable': {'thread_id': 'demo-thread-2', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb542-62c4-6acd-8003-802eb127f677'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'data': '222 A B C D '}, next=('E',), config={'configurable': {'thread_id': 'demo-thread-2', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb542-62c4-6acd-8003-802eb127f677'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-01-06T23:05:09.870254+00:00', parent_config={'configurable': {'thread_id': 'demo-thread-2', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb542-62c2-63e0-8002-83136b8de330'}}, tasks=(PregelTask(id='a5141eec-e9e2-bbbc-f3eb-e1b2d15cbef3', name='E', path=('__pregel_pull', 'E

In [15]:
# Get the latest state snapshot for demo-thread-2
config = {"configurable": {"thread_id": "demo-thread-2"}}
graph.get_state(config)

StateSnapshot(values={'data': '222 A B C D E '}, next=(), config={'configurable': {'thread_id': 'demo-thread-2', 'checkpoint_ns': '', 'checkpoint_id': '1f1796e0-e7a7-6601-8004-71ff4a85ecb8'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-07-06T19:08:22.146595+00:00', parent_config={'configurable': {'thread_id': 'demo-thread-2', 'checkpoint_ns': '', 'checkpoint_id': '1f1796e0-e7a2-63ef-8003-03808d483795'}}, tasks=(), interrupts=())

In [17]:
# Retrieve a specific checkpoint using checkpoint_id, this allows inspection of state at an exact execution point
config = {"configurable": {"thread_id": "demo-thread-2", "checkpoint_id": '1f0eb542-62c2-63e0-8002-83136b8de330'}}
graph.get_state(config)

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': 'demo-thread-2', 'checkpoint_id': '1f0eb542-62c2-63e0-8002-83136b8de330'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())